# ЛР 7. Кураторство активности и происхождение данных

**Сущность:** Отделить сопоставимые точные наблюдения от других endpoint, границ и неизвестных единиц.

**Данные:** activities.csv — восемь искусственных записей; esol.csv — отдельный настоящий открытый корпус.

**Workflow:** Схема → endpoint/мишень/ассей → единицы/отношение → журнал → snapshot hash.

Перед запуском прочитайте [guideline](../guides/lab07.md) и [теорию](../../Module_theory/Module%207_Database.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab07"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Прочитайте условия консервативной политики T1/A1/IC50/=.
2. Преобразуйте только положительные концентрации с известными единицами.
3. Сохраните восемь исходных строк и причины шести исключений, отдельно прочитайте паспорт ESOL.

Ожидаемое: Включены R01/R02 с pIC50=7; в журнале все 8 строк; остальные причины различимы.

In [2]:
from chemistry import curate_activity
import hashlib
with (DATA/"activities.csv").open(encoding="utf-8") as handle:
    records=[curate_activity(row) for row in csv.DictReader(handle)]
table=pd.DataFrame(records);display(table)
included=[r for r in records if r["status"]=="included"]
assert [r["record_id"] for r in included]==["R01","R02"]
assert all(math.isclose(r["pIC50"],7) for r in included)
assert len(records)==8
table.to_csv(OUT/"activity_audit.csv",index=False)
esol=pd.read_csv(DATA/"esol.csv")
print("ESOL отдельно:",len(esol),"строк; measured target присутствует")
assert "measured log solubility in mols per litre" in esol
artifact={"included":2,"excluded":6,"esol_rows":len(esol),"esol_sha256":hashlib.sha256((DATA/"esol.csv").read_bytes()).hexdigest()}


,record_id,molecule_id,target_id,assay_id,endpoint,value,unit,relation,status,pIC50,reason
0,R01,M01,T1,A1,IC50,100,nM,=,included,7.0,teaching rules satisfied
1,R02,M01,T1,A1,IC50,0.1,uM,=,included,7.0,teaching rules satisfied
2,R03,M02,T1,A1,IC50,10,uM,>,excluded,NaN,relation
3,R04,M03,T1,A1,Ki,50,nM,=,excluded,NaN,endpoint
4,R05,M04,T2,A1,IC50,20,nM,=,excluded,NaN,target_id
5,R06,M05,T1,A2,IC50,30,nM,=,excluded,NaN,assay_id
6,R07,M06,T1,A1,IC50,0,nM,=,excluded,NaN,value_or_unit
7,R08,M07,T1,A1,IC50,100,unknown,=,excluded,NaN,value_or_unit


ESOL отдельно: 1128 строк; measured target присутствует


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 7, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: /home/miakhlov/Computational-chemistry_course/Labs/results/lab07/benchmark.json


## 4. Самостоятельная тренировка

1. Сохраните R03 как границу вместо точного числа.
2. Добавьте Ki с тем же числом и объясните, почему это не новый IC50.
3. Спроектируйте pagination/timeout для API, не запрашивая закрытые данные.

[Ответы после попытки](../solutions/lab07.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.